In [1]:
import boto3
import joblib
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
import os

print("sklearn version check:")
import sklearn
print(sklearn.__version__)

iris = load_iris()
X, y = iris.data, iris.target

model = DecisionTreeClassifier()
model.fit(X, y)

joblib.dump(model, "iris-model.pkl")

print("Model saved as iris-model.pkl")


s3 = boto3.client("s3")
bucket = "bucket-for-sagemaker-demo-299029453147-us-east-1-an"

s3.upload_file("iris-model.pkl", bucket, "model-artifacts/iris-model.pkl")

print("Uploaded to S3:", f"s3://{bucket}/model-artifacts/iris-model.pkl")

sklearn version check:
1.4.2
Model saved as iris-model.pkl


Uploaded to S3: s3://bucket-for-sagemaker-demo-299029453147-us-east-1-an/model-artifacts/iris-model.pkl


In [2]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("iris-model.pkl")
    tar.add("inference.py")

print("model.tar.gz created")

model.tar.gz created


In [3]:
import boto3

s3 = boto3.client("s3")
bucket_name = "bucket-for-sagemaker-demo-299029453147-us-east-1-an"
s3.upload_file(
    "model.tar.gz",
    bucket_name,
    "model-artifacts/model.tar.gz"
)

print("Packaged model uploaded to S3")

Packaged model uploaded to S3


In [5]:
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data='s3://bucket-for-sagemaker-demo-299029453147-us-east-1-an/model-artifacts/model.tar.gz',
    role='arn:aws:iam::299029453147:role/SageMakerDomainExecutionRole',
    entry_point='inference.py',
    framework_version='1.4-2',
    py_version='py3'
)

predictor = model.deploy(
    instance_type='ml.t2.medium',
    initial_instance_count=1
)

-

##### Update endpoint

In [ ]:
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data='s3://bucket-for-sagemaker-demo-299029453147-us-east-1-an/model-artifacts/model.tar.gz',
    role='arn:aws:iam::299029453147:role/SageMakerDomainExecutionRole',
    entry_point='inference.py',
    framework_version='1.4-2',
    py_version='py3'
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    endpoint_name='sagemaker-scikit-learn-2026-05-02-13-06-10-669',
    update_endpoint=True
)